# Reentrenamiento HMM - ETTh1 K=5
Entrena Baum-Welch sobre datos RevIN-normalizados (per-window, como en pipeline) y guarda cache.

In [ ]:
import numpy as np
import pandas as pd
import torch
import sys
sys.path.insert(0, '..')
from hmm.baum_welch import baum_welch

In [ ]:
# Cargar train set y normalizar con RevIN (per-window, identico al pipeline)
df = pd.read_csv('../dataset/ETT-small/ETTh1.csv')
n_train = int(len(df) * 0.7)
train_ot = df['OT'].values[:n_train]

seq_len = 96  # Mismo que pipeline

# Crear ventanas non-overlapping, aplicar RevIN a cada una
windows = []
for start in range(0, len(train_ot) - seq_len + 1, seq_len):
    window = train_ot[start:start + seq_len]
    w_mean, w_std = window.mean(), window.std()
    w_std = max(w_std, 1e-5)  # Evitar division por cero
    windows.append((window - w_mean) / w_std)

# Concatenar ventanas RevIN-normalizadas
train_norm = np.concatenate(windows)

print(f'Train original: {len(train_ot)} timesteps')
print(f'Ventanas (stride={seq_len}): {len(windows)} x {seq_len} = {len(train_norm)} timesteps')
print(f'RevIN-normalized: mean={train_norm.mean():.6f}, std={train_norm.std():.6f}')
print(f'Range: [{train_norm.min():.4f}, {train_norm.max():.4f}]')

In [ ]:
# Entrenar HMM (max_iter=500, sobra para K=5 univariado que converge en ~30-80 iters)
result = baum_welch(
    observations=train_norm,
    K=5,
    max_iter=500,
    epsilon=1e-4,
    random_state=42,
    verbose=True
)

print(f'\nconverged: {result["converged"]}')
print(f'n_iter: {result["n_iter"]}')
print(f'log_likelihood: {result["log_likelihood"]:.4f}')
print(f'mu: {result["mu"]}')
print(f'sigma: {result["sigma"]}')
print(f'A diagonal: {np.diag(result["A"])}')
print(f'pi: {result["pi"]}')

# Validar que entreno bien
assert result['converged'], f'NO CONVERGIO en {result["n_iter"]} iters. Aumentar max_iter.'
assert result['log_likelihood'] != float('-inf'), 'log_likelihood es -inf!'
assert np.all(result['sigma'] > 0), 'sigma tiene valores <= 0!'
assert np.allclose(result['A'].sum(axis=1), 1.0, atol=1e-6), 'Filas de A no suman 1!'
assert np.allclose(result['pi'].sum(), 1.0, atol=1e-6), 'pi no suma 1!'
print('\nTodas las validaciones OK')

In [ ]:
# Guardar cache
cache = {
    'A': torch.from_numpy(result['A']).float(),
    'pi': torch.from_numpy(result['pi']).float(),
    'mu': torch.from_numpy(result['mu']).float(),
    'sigma': torch.from_numpy(result['sigma']).float(),
    'log_likelihood': result['log_likelihood'],
    'converged': result['converged'],
    'n_iter': result['n_iter'],
}
torch.save(cache, '../cache/hmm_etth1_K5.pth')
print('Cache guardado en cache/hmm_etth1_K5.pth')

In [ ]:
# Verificar cache guardado
check = torch.load('../cache/hmm_etth1_K5.pth', weights_only=True)
print(f'converged: {check["converged"]}')
print(f'n_iter: {check["n_iter"]}')
print(f'log_likelihood: {check["log_likelihood"]}')
print(f'mu: {check["mu"]}')
print(f'sigma: {check["sigma"]}')
print(f'A:\n{check["A"]}')

assert check['converged'], 'Cache NO convergio!'
assert check['log_likelihood'] != float('-inf'), 'log_likelihood es -inf!'
print('\nCache valido. Listo para usar en exp_plan_a.py')